# Feature Selection

In [5]:
import numpy as np
import pandas as pd 

df = pd.read_csv("./data/07_Feature_Scaling.csv")

In [6]:
print(df.shape)
df.head()

(57130, 102)


,district_name,block_tehsil_name,village_name,latitude,longitude,scheme_type,total_owners_holding,commissioning_year,scheme_depth,scheme_diameter,...,finance_source_Money lender,"finance_source_Money lender, Others","finance_source_Money lender, Own savings",finance_source_Others,"finance_source_Others, Own savings",finance_source_Own savings,finance_source_Owned by Group of farmers,finance_source_Panchayat Owned,geographic_cluster,culturable_command_area_log
0,0.039734,0.001295,0.000018,25.956992,90.799726,1,0.0,0,45.0,150.0,...,0,0,0,0,0,0,0,0,0,1.609438
1,0.020007,0.004428,0.000210,26.355332,90.233585,1,0.0,3,35.0,150.0,...,0,0,0,0,0,0,0,0,0,0.693147
2,0.039734,0.001295,0.000018,25.971319,90.917621,1,0.0,0,60.0,150.0,...,0,0,0,0,0,0,0,0,0,1.609438
3,0.259723,0.016104,0.000210,26.210017,90.160672,1,0.3,1,20.0,80.0,...,0,0,0,0,0,1,0,0,0,1.098612
4,0.259723,0.016104,0.000210,26.208178,90.161543,1,1.0,4,20.0,80.0,...,0,0,0,0,0,1,0,0,0,1.252763


# Data

In [10]:
from sklearn.model_selection import train_test_split

# Separate predictors and target
X = df.drop(columns=["construction_cost"])
y = df["construction_cost"].copy()

# Exclude unwanted predictors without modifying df
excluded_columns = [
    "Unnamed: 0",
    "Total_Outlier_Flags",
    "total_owners_holding"
]

ipc_ipu_columns = [
    col for col in X.columns
    if col.lower().startswith(("ipc_", "ipu_"))
]

columns_to_drop = [
    col for col in excluded_columns + ipc_ipu_columns
    if col in X.columns
]

X = X.drop(columns=columns_to_drop)

# Recreate original train-test split
X_train_scaled, X_test_scaled, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Verification
print("=" * 55)
print("TRAIN-TEST SPLIT VERIFICATION")
print("=" * 55)

print("Original dataset:", df.shape)
print("Excluded columns:", columns_to_drop)

print("\nTraining features:", X_train_scaled.shape)
print("Testing features :", X_test_scaled.shape)
print("Training target  :", y_train.shape)
print("Testing target   :", y_test.shape)

print("\nTotal rows preserved:",
      len(X_train_scaled) + len(X_test_scaled) == len(df))

print("Feature columns match:",
      X_train_scaled.columns.equals(X_test_scaled.columns))

TRAIN-TEST SPLIT VERIFICATION
Original dataset: (57130, 102)
Excluded columns: ['Total_Outlier_Flags', 'total_owners_holding', 'ipc_kharif_season', 'ipc_rabi_season', 'ipc_perennial_crop', 'ipc_other_season', 'ipc_total', 'ipu_kharif_season', 'ipu_rabi_season', 'ipu_perennial_crop', 'ipu_other_season', 'ipu_total']

Training features: (45704, 89)
Testing features : (11426, 89)
Training target  : (45704,)
Testing target   : (11426,)

Total rows preserved: True
Feature columns match: True


# Feature Inspection

In [12]:
# ============================================================
# 1. FEATURE INSPECTION
# ============================================================

import pandas as pd
import numpy as np

# 1. Dataset Dimensions
print("=" * 60)
print("1. DATASET DIMENSIONS")
print("=" * 60)

print("Original Dataset:", df.shape)
print("Training Features:", X_train_scaled.shape)
print("Testing Features:", X_test_scaled.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)


# 2. Data Type Distribution
print("\n2. DATA TYPE DISTRIBUTION")

print(X_train_scaled.dtypes.value_counts())


# 3. Feature Category Identification

binary_features = [
    col for col in X_train_scaled.columns
    if X_train_scaled[col].nunique() == 2
]

one_hot_prefixes = (
    "scheme_type_", "ownership_", "social_status_",
    "gender_", "scheme_status_", "water_distribution_method_",
    "lifting_device_type_", "energy_source_", "well_type_",
    "finance_source_", "captured_in_previous_census_",
    "located_in_command_area_", "only_recharge_ground_water_",
    "drinking_water_purpose_"
)

one_hot_features = [
    col for col in X_train_scaled.columns
    if col.startswith(one_hot_prefixes)
]

ordinal_features = [
    col for col in ["commissioning_year"]
    if col in X_train_scaled.columns
]

frequency_features = [
    col for col in [
        "district_name",
        "block_tehsil_name",
        "village_name"
    ]
    if col in X_train_scaled.columns
]

identified_features = set(
    one_hot_features + ordinal_features + frequency_features
)

other_numerical_features = [
    col for col in X_train_scaled.columns
    if col not in identified_features
]

print("\n3. FEATURE CATEGORY COUNTS")

print("Total Features:", X_train_scaled.shape[1])
print("Binary Features:", len(binary_features))
print("One-Hot Features:", len(one_hot_features))
print("Ordinal Features:", len(ordinal_features))
print("Frequency-Encoded Features:", len(frequency_features))
print("Other Numerical Features:", len(other_numerical_features))


# 4. Excluded Feature Verification

excluded_columns = [
    "Unnamed: 0",
    "Total_Outlier_Flags",
    "total_owners_holding"
]

ipc_ipu_columns = [
    col for col in X_train_scaled.columns
    if col.lower().startswith(("ipc_", "ipu_"))
]

print("\n4. EXCLUDED FEATURE VERIFICATION")

print("Unexpected Excluded Columns:",
      [col for col in excluded_columns
       if col in X_train_scaled.columns])

print("IPC/IPU Columns:", ipc_ipu_columns)


# 5. Data Quality Verification

print("\n5. DATA QUALITY")

print("Training Missing Values:",
      X_train_scaled.isnull().sum().sum())

print("Testing Missing Values:",
      X_test_scaled.isnull().sum().sum())

print("Training Infinite Values:",
      np.isinf(X_train_scaled.select_dtypes(include=np.number)).sum().sum())

print("Testing Infinite Values:",
      np.isinf(X_test_scaled.select_dtypes(include=np.number)).sum().sum())


# 6. Train-Test Consistency

print("\n6. TRAIN-TEST CONSISTENCY")

print("Matching Feature Columns:",
      X_train_scaled.columns.equals(X_test_scaled.columns))

print("Duplicate Feature Names:",
      X_train_scaled.columns.duplicated().sum())

print("Training Rows Preserved:",
      len(X_train_scaled) == 45704)

print("Testing Rows Preserved:",
      len(X_test_scaled) == 11426)


# 7. Feature Information Table

feature_info = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "Data_Type": X_train_scaled.dtypes.astype(str).values,
    "Unique_Values": X_train_scaled.nunique().values,
    "Variance": X_train_scaled.var().values
})

print("\n7. FEATURE INFORMATION")

display(feature_info)

1. DATASET DIMENSIONS
Original Dataset: (57130, 102)
Training Features: (45704, 89)
Testing Features: (11426, 89)
Training Target: (45704,)
Testing Target: (11426,)

2. DATA TYPE DISTRIBUTION
int64      78
float64    11
Name: count, dtype: int64

3. FEATURE CATEGORY COUNTS
Total Features: 89
Binary Features: 76
One-Hot Features: 69
Ordinal Features: 1
Frequency-Encoded Features: 3
Other Numerical Features: 16

4. EXCLUDED FEATURE VERIFICATION
Unexpected Excluded Columns: []
IPC/IPU Columns: []

5. DATA QUALITY
Training Missing Values: 0
Testing Missing Values: 0
Training Infinite Values: 0
Testing Infinite Values: 0

6. TRAIN-TEST CONSISTENCY
Matching Feature Columns: True
Duplicate Feature Names: 0
Training Rows Preserved: True
Testing Rows Preserved: True

7. FEATURE INFORMATION


,Feature,Data_Type,Unique_Values,Variance
0,district_name,float64,33,1.008887e-02
1,block_tehsil_name,float64,169,2.641065e-04
2,village_name,float64,90,2.394346e-07
3,latitude,float64,43872,1.781022e-01
4,longitude,float64,43925,2.106207e+00
...,...,...,...,...
84,finance_source_Own savings,int64,2,1.173991e-01
85,finance_source_Owned by Group of farmers,int64,2,5.287635e-02
86,finance_source_Panchayat Owned,int64,2,1.267453e-03
87,geographic_cluster,int64,3,2.108092e-01


# Correlation Analysis

In [13]:
# ============================================================
# 2. CORRELATION ANALYSIS
# ============================================================

# Combine training predictors and target
train_data = X_train_scaled.copy()
train_data["construction_cost"] = y_train

# Pearson and Spearman correlations
pearson_corr = train_data.corr(method="pearson")["construction_cost"]

spearman_corr = train_data.corr(method="spearman")["construction_cost"]

# Create correlation table
correlation_analysis = pd.DataFrame({
    "Pearson": pearson_corr.drop("construction_cost"),
    "Spearman": spearman_corr.drop("construction_cost")
})

correlation_analysis["Pearson_Abs"] = (
    correlation_analysis["Pearson"].abs()
)

correlation_analysis["Spearman_Abs"] = (
    correlation_analysis["Spearman"].abs()
)

# Sort by Pearson correlation strength
correlation_analysis = correlation_analysis.sort_values(
    by="Pearson_Abs",
    ascending=False
)

# Display results
print("=" * 60)
print("TARGET CORRELATION ANALYSIS")
print("=" * 60)

print("\n1. TOP 20 FEATURES - PEARSON")
display(correlation_analysis.head(20))

print("\n2. TOP 20 FEATURES - SPEARMAN")

display(
    correlation_analysis.sort_values(
        "Spearman_Abs",
        ascending=False
    ).head(20)
)

# Correlation strength distribution
print("\n3. CORRELATION STRENGTH DISTRIBUTION")

for threshold in [0.05, 0.10, 0.30, 0.50, 0.70]:
    count = (
        correlation_analysis["Pearson_Abs"] >= threshold
    ).sum()

    print(f"|Correlation| >= {threshold}: {count}")

# Weakly correlated features
print("\n4. WEAK PEARSON CORRELATION (< 0.05)")

weak_features = correlation_analysis[
    correlation_analysis["Pearson_Abs"] < 0.05
]

print("Count:", len(weak_features))
display(weak_features)

TARGET CORRELATION ANALYSIS

1. TOP 20 FEATURES - PEARSON


,Pearson,Spearman,Pearson_Abs,Spearman_Abs
scheme_depth,0.418613,0.491154,0.418613,0.491154
water_distribution_method_Under ground pipe,0.400797,0.160255,0.400797,0.160255
well_type_Tubewell: Shallow Tube well,-0.373104,-0.461958,0.373104,0.461958
ownership_Govt. Owned,0.373073,0.495269,0.373073,0.495269
well_type_Tubewell: Deep Tube well,0.372914,0.181677,0.372914,0.181677
finance_source_Government fund,0.369536,0.501029,0.369536,0.501029
lifting_device_type_Submersible pump,0.336103,0.416812,0.336103,0.416812
ownership_Owned by individual farmer,-0.305818,-0.434464,0.305818,0.434464
gender_Not applicable,0.305818,0.434464,0.305818,0.434464
social_status_Not applicable,0.305818,0.434464,0.305818,0.434464



2. TOP 20 FEATURES - SPEARMAN


,Pearson,Spearman,Pearson_Abs,Spearman_Abs
finance_source_Government fund,0.369536,0.501029,0.369536,0.501029
ownership_Govt. Owned,0.373073,0.495269,0.373073,0.495269
scheme_depth,0.418613,0.491154,0.418613,0.491154
well_type_Tubewell: Shallow Tube well,-0.373104,-0.461958,0.373104,0.461958
ownership_Owned by individual farmer,-0.305818,-0.434464,0.305818,0.434464
gender_Not applicable,0.305818,0.434464,0.305818,0.434464
social_status_Not applicable,0.305818,0.434464,0.305818,0.434464
gender_Male,-0.298560,-0.425869,0.298560,0.425869
lifting_device_type_Submersible pump,0.336103,0.416812,0.336103,0.416812
well_type_Tubewell: Medium Tube well,0.250449,0.416579,0.250449,0.416579



3. CORRELATION STRENGTH DISTRIBUTION
|Correlation| >= 0.05: 42
|Correlation| >= 0.1: 28
|Correlation| >= 0.3: 10
|Correlation| >= 0.5: 0
|Correlation| >= 0.7: 0

4. WEAK PEARSON CORRELATION (< 0.05)
Count: 47


,Pearson,Spearman,Pearson_Abs,Spearman_Abs
under_utilised_selection_Yes,-0.043322,-0.169004,0.043322,0.169004
water_distribution_method_Other,-0.041777,-0.073411,0.041777,0.073411
water_distribution_method_Open Water Channel (lined / Pucca),0.039328,0.044132,0.039328,0.044132
only_recharge_ground_water,0.037547,0.073709,0.037547,0.073709
water_distribution_method_Surface pipe,-0.034212,0.108782,0.034212,0.108782
social_status_Scheduled-caste,-0.034005,-0.054569,0.034005,0.054569
finance_source_Others,-0.029156,-0.034977,0.029156,0.034977
social_status_Scheduled-tribe,-0.027924,0.024552,0.027924,0.024552
"finance_source_Government fund, Others",0.026899,0.119827,0.026899,0.119827
ownership_Others,-0.025659,-0.029110,0.025659,0.029110


# Multicollinearity Analysis

In [14]:
# ============================================================
# 3. MULTICOLLINEARITY ANALYSIS
# ============================================================

# 1. Calculate predictor correlation matrix
predictor_corr = X_train_scaled.corr(method="pearson")

# 2. Extract upper triangle to avoid duplicate pairs
upper_triangle = predictor_corr.where(
    np.triu(np.ones(predictor_corr.shape), k=1).astype(bool)
)

# 3. Identify highly correlated feature pairs
correlation_threshold = 0.80

high_corr_pairs = (
    upper_triangle.stack()
    .reset_index()
)

high_corr_pairs.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

high_corr_pairs["Absolute_Correlation"] = (
    high_corr_pairs["Correlation"].abs()
)

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["Absolute_Correlation"] >= correlation_threshold
].sort_values(
    "Absolute_Correlation",
    ascending=False
).reset_index(drop=True)

# 4. Display results
print("=" * 65)
print("MULTICOLLINEARITY ANALYSIS")
print("=" * 65)

print("\n1. CORRELATION MATRIX SHAPE")
print(predictor_corr.shape)

print("\n2. HIGHLY CORRELATED FEATURE PAIRS")
print("Threshold:", correlation_threshold)
print("Number of pairs:", len(high_corr_pairs))

display(high_corr_pairs)

# 5. Correlation strength distribution
print("\n3. CORRELATION STRENGTH DISTRIBUTION")

for threshold in [0.70, 0.80, 0.90, 0.95, 0.99]:
    count = (
        upper_triangle.abs() >= threshold
    ).sum().sum()

    print(f"|Correlation| >= {threshold}: {count} pairs")

# 6. Features involved in highly correlated pairs
if not high_corr_pairs.empty:
    involved_features = sorted(set(
        high_corr_pairs["Feature_1"].tolist() +
        high_corr_pairs["Feature_2"].tolist()
    ))

    print("\n4. FEATURES INVOLVED IN HIGH CORRELATION")
    print("Count:", len(involved_features))

    display(pd.DataFrame({
        "Feature": involved_features
    }))
else:
    print("\nNo highly correlated feature pairs found.")

MULTICOLLINEARITY ANALYSIS

1. CORRELATION MATRIX SHAPE
(89, 89)

2. HIGHLY CORRELATED FEATURE PAIRS
Threshold: 0.8
Number of pairs: 23


,Feature_1,Feature_2,Correlation,Absolute_Correlation
0,water_distribution_method_Not applicable,scheme_status_Permanently Not in use,1.000000,1.000000
1,energy_source_Natural,well_type_Tubewell: Artesian Well,1.000000,1.000000
2,energy_source_Manual/animal,lifting_device_type_Manual/animal,1.000000,1.000000
3,energy_source_Not applicable,lifting_device_type_Not applicable,1.000000,1.000000
4,social_status_Not applicable,gender_Not applicable,1.000000,1.000000
5,ownership_Panchayat Owned,finance_source_Panchayat Owned,1.000000,1.000000
6,ownership_Owned by individual farmer,gender_Not applicable,-1.000000,1.000000
7,ownership_Owned by individual farmer,social_status_Not applicable,-1.000000,1.000000
8,ownership_Owned by Group of farmers,finance_source_Owned by Group of farmers,0.996910,0.996910
9,scheme_status_Temporarily Not in use,under_utilised_selection_Unknown,0.994401,0.994401



3. CORRELATION STRENGTH DISTRIBUTION
|Correlation| >= 0.7: 35 pairs
|Correlation| >= 0.8: 23 pairs
|Correlation| >= 0.9: 21 pairs
|Correlation| >= 0.95: 19 pairs
|Correlation| >= 0.99: 11 pairs

4. FEATURES INVOLVED IN HIGH CORRELATION
Count: 27


,Feature
0,energy_source_Manual/animal
1,energy_source_Natural
2,energy_source_Not applicable
3,finance_source_Co-Operative Owned
4,finance_source_Government fund
5,finance_source_Others
6,finance_source_Owned by Group of farmers
7,finance_source_Panchayat Owned
8,gender_Male
9,gender_Not applicable


# Verify perfectly correlated pairs

In [17]:
# ============================================================
# INSPECT FEATURES INVOLVED IN MULTICOLLINEARITY
# ============================================================

involved_features = sorted(set(
    high_corr_pairs["Feature_1"].tolist() +
    high_corr_pairs["Feature_2"].tolist()
))

feature_summary = pd.DataFrame({
    "Feature": involved_features,
    "Unique_Values": [
        X_train_scaled[col].nunique()
        for col in involved_features
    ],
    "Value_0_Count": [
        (X_train_scaled[col] == 0).sum()
        for col in involved_features
    ],
    "Value_1_Count": [
        (X_train_scaled[col] == 1).sum()
        for col in involved_features
    ]
})

display(feature_summary)

,Feature,Unique_Values,Value_0_Count,Value_1_Count
0,energy_source_Manual/animal,2,44819,885
1,energy_source_Natural,2,45688,16
2,energy_source_Not applicable,2,45108,596
3,finance_source_Co-Operative Owned,2,45588,116
4,finance_source_Government fund,2,37499,8205
5,finance_source_Others,2,43147,2557
6,finance_source_Owned by Group of farmers,2,43144,2560
7,finance_source_Panchayat Owned,2,45646,58
8,gender_Male,2,13183,32521
9,gender_Not applicable,2,32975,12729


# Removing redundant columns

In [18]:
# ============================================================
# REMOVE EXACTLY REDUNDANT FEATURES
# ============================================================

columns_to_remove = [
    "water_distribution_method_Not applicable",
    "energy_source_Natural",
    "energy_source_Manual/animal",
    "energy_source_Not applicable",
    "finance_source_Panchayat Owned",
    "social_status_Not applicable",
    "gender_Not applicable"
]

# Verify all selected columns exist
columns_to_remove = [
    col for col in columns_to_remove
    if col in X_train_scaled.columns
]

# Remove from training, testing and predictor datasets
X_train_scaled = X_train_scaled.drop(columns=columns_to_remove)
X_test_scaled = X_test_scaled.drop(columns=columns_to_remove)
X = X.drop(columns=columns_to_remove)

print("Removed Features:", len(columns_to_remove))
print("Removed Columns:")
for col in columns_to_remove:
    print("-", col)

print("\nUpdated Dataset Shapes")
print("X:", X.shape)
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Removed Features: 7
Removed Columns:
- water_distribution_method_Not applicable
- energy_source_Natural
- energy_source_Manual/animal
- energy_source_Not applicable
- finance_source_Panchayat Owned
- social_status_Not applicable
- gender_Not applicable

Updated Dataset Shapes
X: (57130, 82)
X_train_scaled: (45704, 82)
X_test_scaled: (11426, 82)
y_train: (45704,)
y_test: (11426,)


# VIF (Variance Inflation Factor) Analysis

In [21]:
# ============================================================
# DIAGNOSE RANK DEFICIENCY BEFORE VIF
# ============================================================

import numpy as np
import pandas as pd

# Use training predictors only
vif_data = X_train_scaled.copy()

# 1. Identify constant columns
constant_columns = [
    col for col in vif_data.columns
    if vif_data[col].nunique() <= 1
]

# 2. Remove constants from diagnostic copy only
vif_data = vif_data.drop(columns=constant_columns)

# 3. Check matrix rank
matrix_rank = np.linalg.matrix_rank(vif_data.to_numpy())
number_of_features = vif_data.shape[1]

print("=" * 65)
print("VIF RANK DEFICIENCY DIAGNOSIS")
print("=" * 65)

print("Training Matrix Shape:", vif_data.shape)
print("Matrix Rank:", matrix_rank)
print("Number of Features:", number_of_features)
print("Rank Deficiency:", number_of_features - matrix_rank)

print("\nConstant Columns:")
print(constant_columns)

# 4. Identify exact duplicate columns
duplicate_columns = []

for i in range(len(vif_data.columns)):
    for j in range(i + 1, len(vif_data.columns)):
        col_1 = vif_data.iloc[:, i]
        col_2 = vif_data.iloc[:, j]

        if col_1.equals(col_2):
            duplicate_columns.append(
                (vif_data.columns[i], vif_data.columns[j])
            )

print("\nExact Duplicate Column Pairs:")
display(pd.DataFrame(
    duplicate_columns,
    columns=["Feature_1", "Feature_2"]
))

VIF RANK DEFICIENCY DIAGNOSIS
Training Matrix Shape: (45704, 82)
Matrix Rank: 71
Number of Features: 82
Rank Deficiency: 11

Constant Columns:
[]

Exact Duplicate Column Pairs:


,Feature_1,Feature_2


# IDENTIFY LINEARLY DEPENDENT FEATURES

In [22]:
# ============================================================
# IDENTIFY LINEARLY DEPENDENT FEATURES
# ============================================================

import numpy as np
import pandas as pd
from scipy.linalg import qr

vif_data = X_train_scaled.copy()
matrix = vif_data.to_numpy(dtype=float)

# QR decomposition with column pivoting
Q, R, pivot = qr(matrix, mode="economic", pivoting=True)

# Determine numerical rank
tolerance = (
    max(matrix.shape)
    * np.finfo(float).eps
    * abs(R[0, 0])
)

rank = np.sum(np.abs(np.diag(R)) > tolerance)

independent_indices = pivot[:rank]
dependent_indices = pivot[rank:]

independent_columns = vif_data.columns[independent_indices]
dependent_columns = vif_data.columns[dependent_indices]

independent_matrix = matrix[:, independent_indices]

# Explain each dependent feature using independent features
dependency_results = []

for dependent_col in dependent_columns:
    target_vector = vif_data[dependent_col].to_numpy()

    coefficients, _, _, _ = np.linalg.lstsq(
        independent_matrix,
        target_vector,
        rcond=None
    )

    predicted = independent_matrix @ coefficients
    residual = target_vector - predicted

    relevant = np.where(np.abs(coefficients) > 1e-6)[0]

    dependency_results.append({
        "Dependent_Feature": dependent_col,
        "Explained_By": ", ".join(
            f"{independent_columns[i]} ({coefficients[i]:.4f})"
            for i in relevant
        ),
        "Max_Absolute_Residual": np.max(np.abs(residual))
    })

dependency_table = pd.DataFrame(dependency_results)

print("Matrix Rank:", rank)
print("Independent Features:", len(independent_columns))
print("Dependent Features:", len(dependent_columns))

display(dependency_table)

Matrix Rank: 71
Independent Features: 71
Dependent Features: 11


,Dependent_Feature,Explained_By,Max_Absolute_Residual
0,scheme_type,"well_type_Tubewell: Medium Tube well (1.0000),...",1.587619e-14
1,water_distribution_method_Sprinkler,water_distribution_method_Surface pipe (-1.000...,4.800819e-14
2,energy_source_Wind Mills,"energy_source_Electric (-1.0000), energy_sourc...",3.952782e-14
3,finance_source_Co-Operative Owned,"finance_source_Others, Own savings (-1.0000), ...",5.651621e-14
4,under_utilised_selection_Unknown,"under_utilised_selection_No (-1.0000), ownersh...",6.617082e-14
5,social_status_Scheduled-caste,"social_status_Others (-1.0000), social_status_...",1.587404e-14
6,scheme_status_Permanently Not in use,"scheme_status_In-Use (-1.0000), ownership_Owne...",2.828928e-14
7,lifting_device_type_Turbine/Jet pump,lifting_device_type_Centrifugal Pump (-1.0000)...,2.231556e-14
8,well_type_Dugwell: Others,well_type_Tubewell: Medium Tube well (-1.0000)...,4.929252e-14
9,gender_Transgender,"gender_Male (-1.0000), gender_Female (-1.0000)...",9.548379e-15


# Feature Selection Methods — Univariate Analysis

In [23]:
# ============================================================
# 4. UNIVARIATE FEATURE SELECTION
# ============================================================

from sklearn.feature_selection import (
    f_regression,
    mutual_info_regression
)

# 1. F-regression scores
f_scores, f_pvalues = f_regression(
    X_train_scaled,
    y_train
)

# 2. Mutual information scores
mi_scores = mutual_info_regression(
    X_train_scaled,
    y_train,
    random_state=42
)

# 3. Combine results
univariate_results = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "F_Score": f_scores,
    "P_Value": f_pvalues,
    "Mutual_Information": mi_scores
})

# 4. Rank features
univariate_results["F_Rank"] = (
    univariate_results["F_Score"]
    .rank(ascending=False, method="min")
)

univariate_results["MI_Rank"] = (
    univariate_results["Mutual_Information"]
    .rank(ascending=False, method="min")
)

# 5. Display rankings
univariate_results = univariate_results.sort_values(
    "Mutual_Information",
    ascending=False
).reset_index(drop=True)

print("=" * 65)
print("UNIVARIATE FEATURE SELECTION RESULTS")
print("=" * 65)

print("\nTop 20 Features by Mutual Information:")
display(univariate_results.head(20))

print("\nTop 20 Features by F-Regression:")
display(
    univariate_results.sort_values(
        "F_Score", ascending=False
    ).head(20)
)

print("\nFeatures with statistically significant F-test (p < 0.05):")
print((univariate_results["P_Value"] < 0.05).sum())

print("\nTotal Features Evaluated:", len(univariate_results))

UNIVARIATE FEATURE SELECTION RESULTS

Top 20 Features by Mutual Information:


,Feature,F_Score,P_Value,Mutual_Information,F_Rank,MI_Rank
0,block_tehsil_name,1538.973331,0.000000e+00,1.620978,22.0,1.0
1,longitude,1850.030971,0.000000e+00,1.329524,18.0,2.0
2,district_name,1680.886756,0.000000e+00,0.943222,19.0,3.0
3,scheme_depth,9710.269827,0.000000e+00,0.933042,1.0,4.0
4,scheme_diameter,911.175536,3.260289e-198,0.872899,24.0,5.0
5,latitude,396.310963,8.257738e-88,0.857680,28.0,6.0
6,village_name,727.493899,5.555263e-159,0.575716,25.0,7.0
7,culturable_command_area_log,3460.479874,0.000000e+00,0.536428,10.0,8.0
8,culturable_command_area,3384.109143,0.000000e+00,0.529772,11.0,9.0
9,horse_power_of_lifting_device,1630.622086,0.000000e+00,0.438170,20.0,10.0



Top 20 Features by F-Regression:


,Feature,F_Score,P_Value,Mutual_Information,F_Rank,MI_Rank
3,scheme_depth,9710.269827,0.0,0.933042,1.0,4.0
30,water_distribution_method_Under ground pipe,8746.500661,0.0,0.071345,2.0,31.0
12,well_type_Tubewell: Shallow Tube well,7390.896003,0.0,0.294872,3.0,13.0
11,ownership_Govt. Owned,7389.447702,0.0,0.304252,4.0,12.0
31,well_type_Tubewell: Deep Tube well,7382.149087,0.0,0.065061,5.0,32.0
10,finance_source_Government fund,7227.968564,0.0,0.312173,6.0,11.0
14,lifting_device_type_Submersible pump,5820.236196,0.0,0.290278,7.0,15.0
15,ownership_Owned by individual farmer,4715.244966,0.0,0.264219,8.0,16.0
17,gender_Male,4472.447465,0.0,0.258210,9.0,18.0
7,culturable_command_area_log,3460.479874,0.0,0.536428,10.0,8.0



Features with statistically significant F-test (p < 0.05):
64

Total Features Evaluated: 82


## FEATURE IMPORTANCE COMPARISON

In [27]:
# ============================================================
# FEATURE IMPORTANCE COMPARISON
# ============================================================

print("TOP 25 FEATURES")
display(comparison.head(25))

print("\nBOTTOM 20 FEATURES")
display(comparison.tail(20))

print("\nFEATURES RANKED IN BOTTOM 25% BY AT LEAST 4 METHODS")
display(
    comparison[
        comparison["Low_Rank_Count"] >= 4
    ].sort_values(
        ["Low_Rank_Count", "Average_Rank"],
        ascending=[False, False]
    )
)

TOP 25 FEATURES


,Feature,Pearson_Rank,Spearman_Rank,F_Rank,MI_Rank,Random_Forest_Rank,Extra_Trees_Rank,Average_Rank,Low_Rank_Count,Low_Ranked_In_All_Methods
0,scheme_depth,1.0,3.0,1.0,4.0,2.0,3.0,2.333333,0,False
1,ownership_Govt. Owned,4.0,2.0,4.0,12.0,11.0,7.0,6.666667,0,False
2,longitude,20.0,17.0,18.0,2.0,3.0,5.0,10.833333,0,False
3,district_name,21.0,12.0,19.0,3.0,5.0,9.0,11.500000,0,False
4,block_tehsil_name,24.0,15.0,22.0,1.0,7.0,6.0,12.500000,0,False
5,culturable_command_area_log,12.0,19.0,10.0,8.0,13.0,13.0,12.500000,0,False
6,lifting_device_type_Submersible pump,7.0,9.0,7.0,15.0,19.0,24.0,13.500000,0,False
7,culturable_command_area,13.0,19.0,11.0,9.0,14.0,17.0,13.833333,0,False
8,finance_source_Government fund,6.0,1.0,6.0,11.0,25.0,34.0,13.833333,0,False
9,water_distribution_method_Under ground pipe,2.0,31.0,2.0,31.0,15.0,2.0,13.833333,0,False



BOTTOM 20 FEATURES


,Feature,Pearson_Rank,Spearman_Rank,F_Rank,MI_Rank,Random_Forest_Rank,Extra_Trees_Rank,Average_Rank,Low_Rank_Count,Low_Ranked_In_All_Methods
62,well_type_Dugwell: Others,56.0,74.0,52.0,76.0,53.0,60.0,61.833333,2,False
63,well_type_Dugwell: Dug well Pucca,65.0,53.0,60.0,52.0,71.0,71.0,62.000000,3,False
64,energy_source_Wind Mills,66.0,68.0,61.0,74.0,56.0,56.0,63.500000,4,False
65,energy_source_Others,83.0,82.0,77.0,68.0,39.0,41.0,65.000000,4,False
66,"finance_source_Money lender, Others",61.0,56.0,56.0,62.0,80.0,79.0,65.666667,4,False
67,gender_Female,60.0,64.0,55.0,67.0,76.0,75.0,66.166667,4,False
68,finance_source_Co-Operative Owned,75.0,71.0,70.0,60.0,65.0,66.0,67.833333,5,False
69,well_type_Dugwell: Dug well Kutcha,64.0,75.0,59.0,64.0,69.0,76.0,67.833333,5,False
70,located_in_command_area,89.0,73.0,82.0,69.0,47.0,57.0,69.500000,4,False
71,ownership_Co-Operative Owned,74.0,70.0,69.0,75.0,66.0,65.0,69.833333,6,True



FEATURES RANKED IN BOTTOM 25% BY AT LEAST 4 METHODS


,Feature,Pearson_Rank,Spearman_Rank,F_Rank,MI_Rank,Random_Forest_Rank,Extra_Trees_Rank,Average_Rank,Low_Rank_Count,Low_Ranked_In_All_Methods
80,gender_Transgender,81.0,89.0,75.0,76.0,79.0,77.0,79.500000,6,True
81,"finance_source_Bank loan, Money lender",85.0,84.0,79.0,66.0,82.0,81.0,79.500000,6,True
79,finance_source_Bank loan,84.0,88.0,78.0,65.0,78.0,80.0,78.833333,6,True
78,finance_source_Money lender,78.0,81.0,72.0,71.0,81.0,82.0,77.500000,6,True
77,water_distribution_method_Drip,80.0,87.0,74.0,76.0,74.0,72.0,77.166667,6,True
76,"finance_source_Bank loan, Government fund",88.0,65.0,81.0,73.0,73.0,74.0,75.666667,6,True
75,"finance_source_Bank loan, Others",71.0,80.0,66.0,76.0,77.0,78.0,74.666667,6,True
74,"finance_source_Government fund, Money lender",82.0,72.0,76.0,70.0,72.0,73.0,74.166667,6,True
73,well_type_Tubewell: Artesian Well,76.0,77.0,71.0,76.0,75.0,63.0,73.000000,6,True
72,ownership_Panchayat Owned,86.0,66.0,80.0,72.0,64.0,67.0,72.500000,6,True


# Model-Based Feature Importance

In [24]:
# ============================================================
# 5. MODEL-BASED FEATURE IMPORTANCE
# ============================================================

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# 1. Initialize models
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

importance_results = pd.DataFrame({
    "Feature": X_train_scaled.columns
})

# 2. Train models and collect importance
for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)

    importance_results[model_name] = (
        model.feature_importances_
    )

# 3. Calculate average importance
importance_columns = list(models.keys())

importance_results["Average_Importance"] = (
    importance_results[importance_columns].mean(axis=1)
)

# 4. Rank features
importance_results = importance_results.sort_values(
    "Average_Importance",
    ascending=False
).reset_index(drop=True)

importance_results["Rank"] = (
    importance_results.index + 1
)

# 5. Display results
print("=" * 65)
print("MODEL-BASED FEATURE IMPORTANCE")
print("=" * 65)

print("\nTop 20 Features:")
display(importance_results.head(20))

print("\nBottom 20 Features:")
display(importance_results.tail(20))

print("\nTotal Features Evaluated:", len(importance_results))

MODEL-BASED FEATURE IMPORTANCE

Top 20 Features:


,Feature,Random Forest,Extra Trees,Average_Importance,Rank
0,horse_power_of_lifting_device,0.222355,0.161279,0.191817,1
1,scheme_depth,0.134607,0.076548,0.105577,2
2,longitude,0.099176,0.051499,0.075338,3
3,nearest_scheme_distance,0.047620,0.073965,0.060792,4
4,water_distribution_method_Under ground pipe,0.017079,0.099196,0.058138,5
5,latitude,0.065886,0.041489,0.053687,6
6,district_name,0.057401,0.039773,0.048587,7
7,block_tehsil_name,0.047621,0.049212,0.048417,8
8,scheme_diameter,0.054301,0.031354,0.042828,9
9,drinking_water_purpose,0.038569,0.039755,0.039162,10



Bottom 20 Features:


,Feature,Random Forest,Extra Trees,Average_Importance,Rank
62,lifting_device_type_Manual/animal,7.449556e-05,1.279466e-04,1.012211e-04,63
63,well_type_Tubewell: Artesian Well,3.306335e-06,1.546096e-04,7.895798e-05,64
64,ownership_Co-Operative Owned,3.442526e-05,1.212090e-04,7.781712e-05,65
65,finance_source_Co-Operative Owned,3.623903e-05,1.032228e-04,6.973092e-05,66
66,ownership_Panchayat Owned,5.766361e-05,5.732647e-05,5.749504e-05,67
67,social_status_Scheduled-caste,2.506293e-05,4.516447e-05,3.511370e-05,68
68,"finance_source_Money lender, Own savings",2.030990e-05,4.325818e-05,3.178404e-05,69
69,finance_source_Own savings,1.935970e-05,3.457665e-05,2.696817e-05,70
70,well_type_Dugwell: Dug well Pucca,1.341290e-05,2.700195e-05,2.020743e-05,71
71,water_distribution_method_Drip,5.521852e-06,1.975411e-05,1.263798e-05,72



Total Features Evaluated: 82


# Compare Feature Importance Rankings

In [25]:
# ============================================================
# 6. FEATURE IMPORTANCE COMPARISON
# ============================================================

# Prepare rankings from each method
comparison = pd.DataFrame({
    "Feature": X_train_scaled.columns
})

# Correlation ranking
comparison["Pearson_Rank"] = (
    comparison["Feature"]
    .map(pearson_corr.drop("construction_cost").abs().rank(
        ascending=False, method="min"
    ))
)

comparison["Spearman_Rank"] = (
    comparison["Feature"]
    .map(spearman_corr.drop("construction_cost").abs().rank(
        ascending=False, method="min"
    ))
)

# Univariate rankings
comparison["F_Rank"] = (
    comparison["Feature"]
    .map(univariate_results.set_index("Feature")["F_Rank"])
)

comparison["MI_Rank"] = (
    comparison["Feature"]
    .map(univariate_results.set_index("Feature")["MI_Rank"])
)

# Model-based rankings
comparison["Random_Forest_Rank"] = (
    comparison["Feature"]
    .map(importance_results.set_index("Feature")["Random Forest"].rank(
        ascending=False, method="min"
    ))
)

comparison["Extra_Trees_Rank"] = (
    comparison["Feature"]
    .map(importance_results.set_index("Feature")["Extra Trees"].rank(
        ascending=False, method="min"
    ))
)

# Average rank across methods
rank_columns = [
    "Pearson_Rank",
    "Spearman_Rank",
    "F_Rank",
    "MI_Rank",
    "Random_Forest_Rank",
    "Extra_Trees_Rank"
]

comparison["Average_Rank"] = comparison[rank_columns].mean(axis=1)

comparison = comparison.sort_values(
    "Average_Rank"
).reset_index(drop=True)

print("=" * 65)
print("FEATURE IMPORTANCE COMPARISON")
print("=" * 65)

print("\nTop 25 Features Across Methods:")
display(comparison.head(25))

print("\nBottom 20 Features Across Methods:")
display(comparison.tail(20))

FEATURE IMPORTANCE COMPARISON

Top 25 Features Across Methods:


,Feature,Pearson_Rank,Spearman_Rank,F_Rank,MI_Rank,Random_Forest_Rank,Extra_Trees_Rank,Average_Rank
0,scheme_depth,1.0,3.0,1.0,4.0,2.0,3.0,2.333333
1,ownership_Govt. Owned,4.0,2.0,4.0,12.0,11.0,7.0,6.666667
2,longitude,20.0,17.0,18.0,2.0,3.0,5.0,10.833333
3,district_name,21.0,12.0,19.0,3.0,5.0,9.0,11.500000
4,block_tehsil_name,24.0,15.0,22.0,1.0,7.0,6.0,12.500000
5,culturable_command_area_log,12.0,19.0,10.0,8.0,13.0,13.0,12.500000
6,lifting_device_type_Submersible pump,7.0,9.0,7.0,15.0,19.0,24.0,13.500000
7,culturable_command_area,13.0,19.0,11.0,9.0,14.0,17.0,13.833333
8,finance_source_Government fund,6.0,1.0,6.0,11.0,25.0,34.0,13.833333
9,water_distribution_method_Under ground pipe,2.0,31.0,2.0,31.0,15.0,2.0,13.833333



Bottom 20 Features Across Methods:


,Feature,Pearson_Rank,Spearman_Rank,F_Rank,MI_Rank,Random_Forest_Rank,Extra_Trees_Rank,Average_Rank
62,well_type_Dugwell: Others,56.0,74.0,52.0,76.0,53.0,60.0,61.833333
63,well_type_Dugwell: Dug well Pucca,65.0,53.0,60.0,52.0,71.0,71.0,62.000000
64,energy_source_Wind Mills,66.0,68.0,61.0,74.0,56.0,56.0,63.500000
65,energy_source_Others,83.0,82.0,77.0,68.0,39.0,41.0,65.000000
66,"finance_source_Money lender, Others",61.0,56.0,56.0,62.0,80.0,79.0,65.666667
67,gender_Female,60.0,64.0,55.0,67.0,76.0,75.0,66.166667
68,finance_source_Co-Operative Owned,75.0,71.0,70.0,60.0,65.0,66.0,67.833333
69,well_type_Dugwell: Dug well Kutcha,64.0,75.0,59.0,64.0,69.0,76.0,67.833333
70,located_in_command_area,89.0,73.0,82.0,69.0,47.0,57.0,69.500000
71,ownership_Co-Operative Owned,74.0,70.0,69.0,75.0,66.0,65.0,69.833333


# Feature Selection Recommendations

In [26]:
# ============================================================
# 7. FEATURE SELECTION RECOMMENDATIONS
# ============================================================

# Count methods in which a feature appears in the bottom 25%
rank_threshold = int(len(comparison) * 0.75)

comparison["Low_Rank_Count"] = (
    comparison[rank_columns] >= rank_threshold
).sum(axis=1)

comparison["Low_Ranked_In_All_Methods"] = (
    comparison["Low_Rank_Count"] == len(rank_columns)
)

# Identify features consistently ranked low
low_ranked_features = comparison[
    comparison["Low_Rank_Count"] >= 4
].sort_values(
    ["Low_Rank_Count", "Average_Rank"],
    ascending=[False, False]
).reset_index(drop=True)

print("=" * 65)
print("FEATURE SELECTION RECOMMENDATIONS")
print("=" * 65)

print("\nFeatures Ranked in the Bottom 25% by at Least 4 Methods:")
display(low_ranked_features)

print("\nTotal Features Evaluated:", len(comparison))
print("Potential Low-Importance Features:", len(low_ranked_features))
print(
    "Low-Ranked Across All Methods:",
    comparison["Low_Ranked_In_All_Methods"].sum()
)

FEATURE SELECTION RECOMMENDATIONS

Features Ranked in the Bottom 25% by at Least 4 Methods:


,Feature,Pearson_Rank,Spearman_Rank,F_Rank,MI_Rank,Random_Forest_Rank,Extra_Trees_Rank,Average_Rank,Low_Rank_Count,Low_Ranked_In_All_Methods
0,gender_Transgender,81.0,89.0,75.0,76.0,79.0,77.0,79.500000,6,True
1,"finance_source_Bank loan, Money lender",85.0,84.0,79.0,66.0,82.0,81.0,79.500000,6,True
2,finance_source_Bank loan,84.0,88.0,78.0,65.0,78.0,80.0,78.833333,6,True
3,finance_source_Money lender,78.0,81.0,72.0,71.0,81.0,82.0,77.500000,6,True
4,water_distribution_method_Drip,80.0,87.0,74.0,76.0,74.0,72.0,77.166667,6,True
5,"finance_source_Bank loan, Government fund",88.0,65.0,81.0,73.0,73.0,74.0,75.666667,6,True
6,"finance_source_Bank loan, Others",71.0,80.0,66.0,76.0,77.0,78.0,74.666667,6,True
7,"finance_source_Government fund, Money lender",82.0,72.0,76.0,70.0,72.0,73.0,74.166667,6,True
8,well_type_Tubewell: Artesian Well,76.0,77.0,71.0,76.0,75.0,63.0,73.000000,6,True
9,ownership_Panchayat Owned,86.0,66.0,80.0,72.0,64.0,67.0,72.500000,6,True



Total Features Evaluated: 82
Potential Low-Importance Features: 20
Low-Ranked Across All Methods: 11


# Check feature frequency

In [28]:
low_ranked_candidates = comparison.loc[
    comparison["Low_Ranked_In_All_Methods"], "Feature"
].tolist()

frequency_results = pd.DataFrame({
    "Feature": low_ranked_candidates,
    "Count": [
        int(X_train_scaled[col].sum())
        for col in low_ranked_candidates
    ],
    "Percentage": [
        X_train_scaled[col].mean() * 100
        for col in low_ranked_candidates
    ]
})

display(frequency_results.sort_values("Percentage"))

,Feature,Count,Percentage
10,"finance_source_Bank loan, Money lender",2,0.004376
8,finance_source_Bank loan,6,0.013128
7,finance_source_Money lender,9,0.019692
9,gender_Transgender,11,0.024068
2,well_type_Tubewell: Artesian Well,16,0.035008
3,"finance_source_Government fund, Money lender",16,0.035008
5,"finance_source_Bank loan, Government fund",22,0.048136
4,"finance_source_Bank loan, Others",33,0.072204
6,water_distribution_method_Drip,52,0.113776
1,ownership_Panchayat Owned,58,0.126904


In [30]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_scaled, y_train,
    test_size=0.20,
    random_state=42
)

low_ranked_candidates = comparison.loc[
    comparison["Low_Ranked_In_All_Methods"], "Feature"
].tolist()

model_all = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)

model_reduced = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)

model_all.fit(X_fit, y_fit)
model_reduced.fit(X_fit.drop(columns=low_ranked_candidates), y_fit)

rmse_all = root_mean_squared_error(
    y_val, model_all.predict(X_val)
)

rmse_reduced = root_mean_squared_error(
    y_val,
    model_reduced.predict(X_val.drop(columns=low_ranked_candidates))
)

print(f"All 82 features RMSE: {rmse_all:.4f}")
print(f"Without 11 features RMSE: {rmse_reduced:.4f}")

All 82 features RMSE: 304058.9568
Without 11 features RMSE: 300715.9965


# feature removal

In [31]:
selected_features = [
    col for col in X_train_scaled.columns
    if col not in low_ranked_candidates
]

X_train_selected = X_train_scaled[selected_features].copy()
X_test_selected = X_test_scaled[selected_features].copy()

print("FEATURE SELECTION SUMMARY")
print("Original Features:", X_train_scaled.shape[1])
print("Selected Features:", X_train_selected.shape[1])
print("Removed Features:", len(low_ranked_candidates))
print("Training Shape:", X_train_selected.shape)
print("Testing Shape:", X_test_selected.shape)

FEATURE SELECTION SUMMARY
Original Features: 82
Selected Features: 71
Removed Features: 11
Training Shape: (45704, 71)
Testing Shape: (11426, 71)


# Save Dataset

In [37]:
# Combine selected features with the target
train_selected = X_train_selected.copy()
train_selected["construction_cost"] = y_train

test_selected = X_test_selected.copy()
test_selected["construction_cost"] = y_test

# Save datasets
train_selected.to_csv(
    "./data/08_Feature_Selected_Train.csv", index=False
)

test_selected.to_csv(
    "./data/08_Feature_Selected_Test.csv", index=False
)

print("Training shape:", train_selected.shape)
print("Testing shape:", test_selected.shape)
print("Datasets saved successfully.")

Training shape: (45704, 72)
Testing shape: (11426, 72)
Datasets saved successfully.
